In [0]:
# ============================================================
# Notebook 4: Feature Extraction & Selection
# ATP Tennis Dataset (2000-2026)
# ============================================================

storage_account = "tennisdatalake60105845"
storage_key = "LjZoIlf5yGSIHxM/9GIXHx5jTq8VNvMOqWjuLZ54krBy3gn+BN7pg8q5/4UM8dgtAEwIMaTNyIYl+AStqzCGpQ=="
spark.conf.set(
    f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
    storage_key
)

# Load cleaned data
df = spark.read.format("delta").table("amazon_dbx_60105845.default.atp_tennis_clean")

print(f" Loaded {df.count()} records")
df.show(3)

 Loaded 67460 records
+---------------+----------+----------+-------+-------+----------+-------+----------+-----------+--------+------+------+-----+-----+-----+-----+-----------+-----------------+----+-----+
|     Tournament|      Date|    Series|  Court|Surface|     Round|Best_of|  Player_1|   Player_2|  Winner|Rank_1|Rank_2|Pts_1|Pts_2|Odd_1|Odd_2|      Score|winner_is_player1|year|month|
+---------------+----------+----------+-------+-------+----------+-------+----------+-----------+--------+------+------+-----+-----+-----+-----+-----------+-----------------+----+-----+
|   Chennai Open|2013-12-30|    ATP250|Outdoor|   Hard| 1st Round|      3|Smyczek T.|    Lu Y.H.| Lu Y.H.|    89|    65|  600|  730| 2.37| 1.53|    4-6 2-6|                0|2013|   12|
|Australian Open|2014-01-24|Grand Slam|Outdoor|   Hard|Semifinals|      5|  Nadal R.| Federer R.|Nadal R.|     1|     6|13130| 4355| 1.57| 2.37|7-6 6-3 6-3|                1|2014|    1|
|     Copa Claro|2014-02-11|    ATP250|Outdoor| 

In [0]:
# ============================================================
# Feature Extraction
# ============================================================
from pyspark.sql.functions import col, when, abs as spark_abs

# 1. Rank difference (higher ranked player advantage)
df_features = df.withColumn("rank_diff", col("Rank_1") - col("Rank_2"))

# 2. Points difference
df_features = df_features.withColumn("pts_diff", col("Pts_1") - col("Pts_2"))

# 3. Odds difference (betting market signal)
df_features = df_features.withColumn("odds_diff", col("Odd_1") - col("Odd_2"))

# 4. Is player 1 higher ranked (lower number = better rank)
df_features = df_features.withColumn("p1_higher_ranked",
    when(col("Rank_1") < col("Rank_2"), 1).otherwise(0))

# 5. Is Grand Slam
df_features = df_features.withColumn("is_grand_slam",
    when(col("Series") == "Grand Slam", 1).otherwise(0))

# 6. Is Hard court
df_features = df_features.withColumn("is_hard_court",
    when(col("Surface") == "Hard", 1).otherwise(0))

# 7. Is Clay court
df_features = df_features.withColumn("is_clay_court",
    when(col("Surface") == "Clay", 1).otherwise(0))

# 8. Is best of 5
df_features = df_features.withColumn("is_best_of_5",
    when(col("Best_of") == 5, 1).otherwise(0))

print(" Features created!")
df_features.select("rank_diff", "pts_diff", "odds_diff", "p1_higher_ranked", 
                   "is_grand_slam", "is_hard_court", "is_clay_court", 
                   "is_best_of_5", "winner_is_player1").show(5)

 Features created!
+---------+--------+-------------------+----------------+-------------+-------------+-------------+------------+-----------------+
|rank_diff|pts_diff|          odds_diff|p1_higher_ranked|is_grand_slam|is_hard_court|is_clay_court|is_best_of_5|winner_is_player1|
+---------+--------+-------------------+----------------+-------------+-------------+-------------+------------+-----------------+
|       24|    -130| 0.8400000000000001|               0|            0|            1|            0|           0|                0|
|       -5|    8775|               -0.8|               1|            1|            1|            0|           1|                1|
|      -54|     264|               -1.0|               1|            0|            0|            1|           0|                1|
|      121|   -1190|               0.97|               0|            0|            1|            0|           0|                0|
|      -15|    3710|-3.3200000000000003|               1|       

In [0]:
# ============================================================
# Feature Selection & Save
# ============================================================
from pyspark.sql.functions import col

# Select final features for ML
feature_cols = [
    "rank_diff",        # rank difference between players
    "pts_diff",         # points difference
    "odds_diff",        # betting odds difference
    "p1_higher_ranked", # is player 1 better ranked
    "is_grand_slam",    # is it a grand slam
    "is_hard_court",    # surface type
    "is_clay_court",    # surface type
    "is_best_of_5",     # match format
    "Rank_1",           # absolute rank of player 1
    "Rank_2",           # absolute rank of player 2
    "year",             # year of match
    "month",            # month of match
    "winner_is_player1" # TARGET VARIABLE
]

df_final = df_features.select(feature_cols).dropna()

print(f"Final feature set: {len(feature_cols)-1} features + 1 target")
print(f"Records after dropping nulls: {df_final.count()}")

# Save to curated zone
df_final.write.format("delta") \
    .mode("overwrite") \
    .save(f"abfss://curated@{storage_account}.dfs.core.windows.net/tennis/features/")

# Also register as table
df_final.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("amazon_dbx_60105845.default.atp_tennis_features")

print(" Features saved to curated zone!")
print(" Features registered as table in catalog!")

Final feature set: 12 features + 1 target
Records after dropping nulls: 67433
 Features saved to curated zone!
 Features registered as table in catalog!
